# 04 — Power & calibration at realistic site sizes (synthetic)

Before reporting any significant difference from the paired bootstrap on real data, calibrate the test at *your* site Ns. This notebook:

1. Runs the calibration sim at sample sizes close to the actual US / San Borja / Tsimane' counts.
2. Sweeps $\Delta\rho$ to estimate power vs effect size.
3. Sweeps `n_items` to show coverage-vs-sample-size.

Recommended config matches Globalized-Music after the d' filter (rough estimates — adjust to match your actual loaded Ns from notebook 05).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd(); ROOT = HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python import power_simulation_paired_bootstrap

## Realistic Ns and reliabilities

Edit these to match what notebook 05 reports for your actual data. Defaults below are ballpark Globalized-Music after d' >= 2.0 filter.

In [ ]:
N_US = 40
N_SB = 30
N_TSI = 30
N_ITEMS = 50
TARGET_REL = 0.6        # within-group SB-corrected reliability you observe
BASE_RATE = 0.7
ITEM_SPREAD = 0.15
N_REPS = 200            # bump to 500-1000 for a final report
N_BOOT = 1000
ALPHA = 0.05

## Type-I error at H0 (all three true correlations equal at 0.5)

In [ ]:
h0 = power_simulation_paired_bootstrap(
    n_reps=N_REPS, n_a=N_US, n_b=N_SB, n_c=N_TSI, n_items=N_ITEMS,
    rho=(0.5, 0.5, 0.5), rel=(TARGET_REL,)*3,
    base_rate=BASE_RATE, item_spread=ITEM_SPREAD,
    n_boot=N_BOOT, alpha=ALPHA, seed=101, progress_every=50,
)
print('\nType-I summary:')
print(f'  recentered-null reject rates: {h0.reject_null.round(3)} (target ~ {ALPHA})')
print(f'  straddle-zero   reject rates: {h0.reject_straddle.round(3)}')
print(f'  CI coverage                 : {h0.coverage.round(3)} (target ~ {1-ALPHA})')

## Power sweep — vary the effect size $\Delta\rho$

Holds $\rho_{AC}=\rho_{BC}=0.40$, increases $\rho_{AB}$ from 0.40 to 0.75. Power for AB-vs-AC and AB-vs-BC should rise; AC-vs-BC should hover at $\alpha$ (it's still under H0).

In [ ]:
deltas = [0.0, 0.10, 0.20, 0.30, 0.40]
rho_base = 0.40
power_grid = np.zeros((len(deltas), 3))

for i, d in enumerate(deltas):
    res = power_simulation_paired_bootstrap(
        n_reps=N_REPS, n_a=N_US, n_b=N_SB, n_c=N_TSI, n_items=N_ITEMS,
        rho=(rho_base + d, rho_base, rho_base), rel=(TARGET_REL,)*3,
        base_rate=BASE_RATE, item_spread=ITEM_SPREAD,
        n_boot=N_BOOT, alpha=ALPHA, seed=200 + i, progress_every=0,
        verbose=False,
    )
    power_grid[i] = res.reject_null
    print(f'  delta_rho = {d:.2f}:  reject(AB-AC)={res.reject_null[0]:.3f}  '
          f'reject(AB-BC)={res.reject_null[1]:.3f}  reject(AC-BC)={res.reject_null[2]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['AB vs AC', 'AB vs BC', 'AC vs BC (null)']
colors = ['#4060b0', '#40a060', '#b04040']
for k in range(3):
    ax.plot(deltas, power_grid[:, k], 'o-', color=colors[k], label=labels[k])
ax.axhline(ALPHA, color='k', lw=0.8, ls='--', alpha=0.5, label=f'alpha={ALPHA}')
ax.axhline(0.8, color='gray', lw=0.8, ls=':', alpha=0.5, label='conventional 80%')
ax.set_xlabel('delta rho (effect size)'); ax.set_ylabel('rejection rate')
ax.set_title(f'Power vs effect size  (N={N_US}/{N_SB}/{N_TSI}, items={N_ITEMS}, rel={TARGET_REL})')
ax.legend(loc='center right')
ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

## n_items sweep (optional)

How much does the test depend on shared-item count? Holds Ns and $\Delta\rho=0.25$.

In [ ]:
n_items_grid = [20, 40, 60, 80]
power_items = np.zeros((len(n_items_grid), 3))
for i, ni in enumerate(n_items_grid):
    res = power_simulation_paired_bootstrap(
        n_reps=N_REPS, n_a=N_US, n_b=N_SB, n_c=N_TSI, n_items=ni,
        rho=(0.65, 0.40, 0.40), rel=(TARGET_REL,)*3,
        base_rate=BASE_RATE, item_spread=ITEM_SPREAD,
        n_boot=N_BOOT, alpha=ALPHA, seed=300 + i, progress_every=0,
        verbose=False,
    )
    power_items[i] = res.reject_null
    print(f'  n_items={ni:3d}:  AB-AC={res.reject_null[0]:.3f}  AB-BC={res.reject_null[1]:.3f}  '
          f'AC-BC={res.reject_null[2]:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
for k in range(3):
    ax.plot(n_items_grid, power_items[:, k], 'o-', color=colors[k], label=labels[k])
ax.axhline(ALPHA, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('shared items'); ax.set_ylabel('rejection rate')
ax.set_title(f'Power vs n_items  (delta_rho=0.25)')
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()

**How to use the result.**

- If `reject_null` under H0 is far from $\alpha$ at your Ns, the test is mis-calibrated and any p-value from real data should be reported with that caveat.
- The Power-vs-$\Delta\rho$ curve tells you the smallest effect size you can plausibly detect at 80% power. If your real-data $\Delta\rho$ is much smaller, a non-significant result is uninformative.
- Drop these calibration numbers in the paper alongside the real-data p.